In [ ]:
# 한국어 형태소 분석을 위한 KoNLPy 설치
!pip install konlpy

In [ ]:
# 데이터셋이 Data 폴더에 있는지 확인 (CNN 폴더 기준 또는 프로젝트 루트 기준)
import os
import urllib.request

train_candidates = [
    '../Data/NSMC/ratings_train.txt',
    'Data/NSMC/ratings_train.txt',
    'ratings_train.txt'
]
test_candidates = [
    '../Data/NSMC/ratings_test.txt',
    'Data/NSMC/ratings_test.txt',
    'ratings_test.txt'
]

train_path = None
for path in train_candidates:
    if os.path.exists(path):
        train_path = path
        print(f"Found train dataset at: {train_path}")
        break

test_path = None
for path in test_candidates:
    if os.path.exists(path):
        test_path = path
        print(f"Found test dataset at: {test_path}")
        break

if not train_path:
    train_url = 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt'
    train_path = 'ratings_train.txt'
    print("Downloading ratings_train.txt...")
    urllib.request.urlretrieve(train_url, train_path)

if not test_path:
    test_url = 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt'
    test_path = 'ratings_test.txt'
    print("Downloading ratings_test.txt...")
    urllib.request.urlretrieve(test_url, test_path)

print("Setup complete!")
print(f"Train data path: {train_path}, Size: {os.path.getsize(train_path)} bytes")
print(f"Test data path: {test_path}, Size: {os.path.getsize(test_path)} bytes")

In [ ]:
# 데이터 전처리, 토큰화 및 단어 사전 구축
import pandas as pd
import numpy as np
from konlpy.tag import Okt
from collections import Counter

# 1. 데이터 로드
print("Loading datasets...")
train_data = pd.read_csv(train_path, sep='\t')
test_data = pd.read_csv(test_path, sep='\t')
print(f"Original Train size: {len(train_data)}, Test size: {len(test_data)}")

# 2. 중복 데이터 및 결측치 제거 & Train/Validation 분리
train_data.drop_duplicates(subset=['document'], inplace=True)
train_data.dropna(how='any', inplace=True)
test_data.drop_duplicates(subset=['document'], inplace=True)
test_data.dropna(how='any', inplace=True)

# 훈련 데이터에서 10%를 검증 데이터(Validation)로 분리 (난수 시드 42 고정)
val_data = train_data.sample(frac=0.1, random_state=42)
train_data = train_data.drop(val_data.index)
print(f"Cleaned & Split - Train size: {len(train_data)}, Validation size: {len(val_data)}, Test size: {len(test_data)}")

# 3. Okt 형태소 분석기를 이용한 토큰화 및 불용어 제거
okt = Okt()
stopwords = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다']

def tokenize(text):
    # 형태소 분석 및 어근 추출
    tokens = okt.morphs(text, stem=True)
    # 불용어 제거
    tokens = [word for word in tokens if word not in stopwords]
    return tokens

# 훈련, 검증, 테스트 데이터 토큰화 수행
print("Tokenizing train data... (This might take 2-3 minutes on Colab)")
train_data['tokenized'] = train_data['document'].apply(tokenize)
print("Tokenizing validation data...")
val_data['tokenized'] = val_data['document'].apply(tokenize)
print("Tokenizing test data...")
test_data['tokenized'] = test_data['document'].apply(tokenize)

# 4. 단어 사전(Vocabulary) 구축 (데이터 누수 방지를 위해 Train 데이터만으로 구축)
print("Building vocabulary...")
all_words = []
for tokens in train_data['tokenized']:
    all_words.extend(tokens)

word_counts = Counter(all_words)
# 2회 이상 등장한 단어들만 사전에 유지
vocab = ['<pad>', '<unk>'] + [word for word, count in word_counts.items() if count >= 2]
word_to_index = {word: idx for idx, word in enumerate(vocab)}
index_to_word = {idx: word for idx, word in enumerate(vocab)}

print(f"Vocabulary size: {len(vocab)}")

# 5. 샘플 텍스트 인코딩 결과 확인
sample_text = "이 영화 정말 재미있고 감동적이네요!"
sample_tokens = tokenize(sample_text)
sample_encoded = [word_to_index.get(w, word_to_index['<unk>']) for w in sample_tokens]
print(f"\nSample text: '{sample_text}'")
print(f"Tokens: {sample_tokens}")
print(f"Encoded: {sample_encoded}")

In [ ]:
# PyTorch Dataset 및 DataLoader 구현
import torch
from torch.utils.data import Dataset, DataLoader

class NSMCDataset(Dataset):
    def __init__(self, tokenized_texts, labels, word_to_index, max_len=30):
        # 판다스 시리즈인 경우 리스트로 변환
        self.tokenized_texts = tokenized_texts.tolist() if hasattr(tokenized_texts, 'tolist') else list(tokenized_texts)
        self.labels = labels.tolist() if hasattr(labels, 'tolist') else list(labels)
        self.word_to_index = word_to_index
        self.max_len = max_len
        self.pad_idx = word_to_index.get('<pad>', 0)
        self.unk_idx = word_to_index.get('<unk>', 1)
        
    def __len__(self):
        return len(self.tokenized_texts)
        
    def __getitem__(self, idx):
        tokens = self.tokenized_texts[idx]
        label = self.labels[idx]
        
        # 단어를 고유 번호로 변환 (모르는 단어는 <unk>로 치환)
        encoded = [self.word_to_index.get(token, self.unk_idx) for token in tokens]
        
        # 문장 길이 규격화 (패딩 혹은 자르기)
        if len(encoded) < self.max_len:
            encoded += [self.pad_idx] * (self.max_len - len(encoded))
        else:
            encoded = encoded[:self.max_len]
            
        return torch.LongTensor(encoded), torch.tensor(label, dtype=torch.long)

# 1. 최대 문장 길이 정의
max_len = 30

# 2. 데이터셋 객체 생성 (훈련용, 검증용, 테스트용 분리)
train_dataset = NSMCDataset(train_data['tokenized'], train_data['label'], word_to_index, max_len)
val_dataset = NSMCDataset(val_data['tokenized'], val_data['label'], word_to_index, max_len)
test_dataset = NSMCDataset(test_data['tokenized'], test_data['label'], word_to_index, max_len)

# 3. 데이터로더 생성 (배치 크기 = 64)
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train Dataset size: {len(train_dataset)}")
print(f"Validation Dataset size: {len(val_dataset)}")
print(f"Test Dataset size: {len(test_dataset)}")

# 4. 검증: 첫 번째 배치를 추출하여 차원 확인
for x_batch, y_batch in train_loader:
    print(f"\nFirst batch inputs shape (batch_size, max_len): {x_batch.shape}")
    print(f"First batch labels shape (batch_size): {y_batch.shape}")
    print(f"Sample token IDs of the first document:\n{x_batch[0]}")
    print(f"Sample label of the first document: {y_batch[0].item()}")
    break

In [ ]:
# 감정 분석을 위한 CNN 모델 아키텍처 정의
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN_Sentiment(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_filters, filter_sizes, output_dim, dropout, padding_idx=0):
        super().__init__()
        
        # 1. 임베딩 레이어 (패딩 토큰의 그래디언트 학습은 제외하기 위해 padding_idx 지정)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        
        # 2. Conv2d와 ModuleList를 활용한 합성곱 레이어 정의
        self.convs = nn.ModuleList([
            nn.Conv2d(in_channels=1, 
                      out_channels=n_filters, 
                      kernel_size=(fs, embedding_dim)) 
            for fs in filter_sizes
        ])
        
        # 3. 완전연결 레이어 (FC Layer) 및 드롭아웃
        self.fc = nn.Linear(len(filter_sizes) * n_filters, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text):
        # text 형태: [batch_size, seq_len]
        
        embedded = self.embedding(text)
        # embedded 형태: [batch_size, seq_len, embedding_dim]
        
        # Conv2d 연산을 위해 채널 차원(1채널) 추가
        embedded = embedded.unsqueeze(1)
        # embedded 형태: [batch_size, 1, seq_len, embedding_dim]
        
        # 합성곱 및 맥스 풀링 연산
        pooled = []
        for conv in self.convs:
            # conv(embedded) 형태: [batch_size, n_filters, seq_len - fs + 1, 1]
            conved = F.relu(conv(embedded)).squeeze(3)
            # conved 형태: [batch_size, n_filters, seq_len - fs + 1]
            
            # 문장 시퀀스 차원(dim=2)에 대해 맥스 풀링 수행
            pool = torch.max(conved, dim=2)[0]
            # pool 형태: [batch_size, n_filters]
            pooled.append(pool)
            
        # 추출된 다양한 크기 필터들의 특징들을 하나로 결합
        cat = self.dropout(torch.cat(pooled, dim=1))
        # cat 형태: [batch_size, len(filter_sizes) * n_filters]
        
        return self.fc(cat)

# 1. 모델 하이퍼파라미터 정의
vocab_size = len(vocab) if 'vocab' in globals() else 1000  # 테스트용 예외 처리
embedding_dim = 128
n_filters = 100
filter_sizes = [3, 4, 5]
output_dim = 2
dropout = 0.5

# 2. 모델 인스턴스화
model = CNN_Sentiment(vocab_size, embedding_dim, n_filters, filter_sizes, output_dim, dropout) # 이 때 init 자동 실행
print("Model Architecture:")
print(model)

# 3. 가상 텐서를 활용한 모델 출력 차원 검증
dummy_input = torch.randint(0, vocab_size, (2, 30)) # [배치 크기=2, 시퀀스 길이=30]
dummy_output = model(dummy_input)
print(f"\nDummy input shape: {dummy_input.shape}")
print(f"Dummy output shape (batch_size, output_dim): {dummy_output.shape}")
assert dummy_output.shape == (2, 2), "Output shape verification failed!"
print("Model output shape verification passed!")

In [ ]:
# GPU 학습 루프 및 검증
import torch.optim as optim
import json
import os

# 1. 연산 장치 설정 (GPU 사용 가능 여부 자동 체크)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2. 모델을 지정된 장치로 전송
model = model.to(device)

# 3. 손실 함수 및 옵티마이저 정의 (Adam, lr=0.001)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. 학습 매개변수 설정
epochs = 5
best_val_acc = 0.0

# 5. 학습 추이 시각화를 위한 로그 기록 변수 정의
train_losses = []
train_accs = []
val_losses = []
val_accs = []

# 학습 결과를 파일로 저장하기 위한 히스토리 리스트
metrics_history = []

print("\nStarting model training...\n")
for epoch in range(epochs):
    # --- 학습 단계 ---
    model.train()
    epoch_train_loss = 0.0
    epoch_train_correct = 0
    total_train_samples = 0
    
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # 순전파 연산
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        # 역전파 및 가중치 최적화
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 평가지표 누적 기록
        epoch_train_loss += loss.item() * batch_x.size(0)
        preds = outputs.argmax(dim=1)
        epoch_train_correct += (preds == batch_y).sum().item()
        total_train_samples += batch_x.size(0)
        
    # 에폭 단위 학습 지표 계산
    train_loss = epoch_train_loss / total_train_samples
    train_acc = epoch_train_correct / total_train_samples
    
    # --- 검증 단계 (Validation Set 사용) ---
    model.eval()
    epoch_val_loss = 0.0
    epoch_val_correct = 0
    total_val_samples = 0
    
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            epoch_val_loss += loss.item() * batch_x.size(0)
            preds = outputs.argmax(dim=1)
            epoch_val_correct += (preds == batch_y).sum().item()
            total_val_samples += batch_x.size(0)
            
    val_loss = epoch_val_loss / total_val_samples
    val_acc = epoch_val_correct / total_val_samples
    
    # 학습 역사 기록
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    print(f"Epoch [{epoch+1}/{epochs}] - "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%")
          
    # 에폭 메트릭 기록 저장
    metrics_history.append({
        "epoch": epoch + 1,
        "split": "train",
        "loss": float(train_loss),
        "accuracy": float(train_acc)
    })
    metrics_history.append({
        "epoch": epoch + 1,
        "split": "validation",
        "loss": float(val_loss),
        "accuracy": float(val_acc)
    })
    
    # 체크포인트: 검증 정확도가 갱신되면 가중치 저장
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_cnn_model.pt')
        print(f"  --> Found new best model! Saved weight to 'best_cnn_model.pt'")

# 학습이 끝나면 metrics.json 파일에 기록 저장 (우선 훈련/검증 결과만 저장)
metrics_path = 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_history, f, indent=2, ensure_ascii=False)

print(f"\nTraining complete! Best Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"Saved training metrics to {os.path.abspath(metrics_path)}")

In [ ]:
# 학습 및 검증 진행 추이 시각화
import matplotlib.pyplot as plt

# 1. 에폭 축 정의 (1부터 시작)
epochs_range = range(1, len(train_losses) + 1)

# 2. 훈련 히스토리 그래프 그리기
plt.figure(figsize=(12, 5))

# 서브플롯 1: 손실(Loss) 추이
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label='Train Loss', marker='o')
plt.plot(epochs_range, val_losses, label='Val Loss', marker='s')
plt.title('Training & Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# 서브플롯 2: 정확도(Accuracy) 추이
plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_accs, label='Train Acc', marker='o')
plt.plot(epochs_range, val_accs, label='Val Acc', marker='s')
plt.title('Training & Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ipywidgets를 활용한 실시간 감정 분석 예측 테스트 위젯
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. 새로운 문장에 대한 감정 예측 수행 함수
def predict_sentiment(text, model, word_to_index, max_len=30, device='cpu'):
    # 입력 문장 토큰화
    tokens = tokenize(text)
    
    # 단어를 번호 인덱스로 변환
    unk_idx = word_to_index.get('<unk>', 1)
    pad_idx = word_to_index.get('<pad>', 0)
    encoded = [word_to_index.get(token, unk_idx) for token in tokens]
    
    # 문장 규격 맞추기 (패딩 및 자르기)
    if len(encoded) < max_len:
        encoded += [pad_idx] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]
        
    # 배치 차원을 추가하여 텐서로 변환 [1, seq_len]
    tensor = torch.LongTensor(encoded).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs = F.softmax(logits, dim=1).squeeze(0)
        
    pred_label = probs.argmax().item()
    confidence = probs[pred_label].item()
    pos_prob = probs[1].item()
    neg_prob = probs[0].item()
    
    return pred_label, confidence, pos_prob, neg_prob

# 2. 저장해 둔 최고 성능 가중치 로드
if os.path.exists('best_cnn_model.pt'):
    print("Loading best model weights from 'best_cnn_model.pt'...")
    model.load_state_dict(torch.load('best_cnn_model.pt', map_location=device))
else:
    print("Warning: 'best_cnn_model.pt' not found. Using current model weights.")

# 3. 최종 모델 성능 평가 (Test Set 평가)
print("\nEvaluating the best model on the Test Set...")
model.eval()
test_loss = 0.0
test_correct = 0
total_test_samples = 0
criterion = nn.CrossEntropyLoss()

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        test_loss += loss.item() * batch_x.size(0)
        preds = outputs.argmax(dim=1)
        test_correct += (preds == batch_y).sum().item()
        total_test_samples += batch_x.size(0)

final_test_loss = test_loss / total_test_samples
final_test_acc = test_correct / total_test_samples
print(f"Test Loss: {final_test_loss:.4f}, Test Accuracy: {final_test_acc*100:.2f}%")

# 최종 테스트 결과를 metrics.json에 추가 저장
metrics_path = 'metrics.json'
if os.path.exists(metrics_path):
    try:
        with open(metrics_path, 'r', encoding='utf-8') as f:
            history = json.load(f)
    except Exception:
        history = []
else:
    history = []

# 기존 기록에 테스트 평가 기록 덧붙이기
history.append({
    "split": "test",
    "test_file": "Data/NSMC/ratings_test.txt",
    "loss": float(final_test_loss),
    "accuracy": float(final_test_acc)
})

with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2, ensure_ascii=False)
print(f"Added test metrics to {metrics_path}")

# 4. ipywidgets를 활용한 UI 위젯 디자인
text_input = widgets.Textarea(
    value="이 영화 진짜 시간 가는 줄 모르고 봤어요! 대박 강추!",
    placeholder="여기에 영화 리뷰를 작성하세요...",
    description="리뷰 입력:",
    layout=widgets.Layout(width='90%', height='100px')
)

analyze_button = widgets.Button(
    description="감정 분석 실행",
    button_style='success',  # 초록색 버튼
    tooltip='입력된 영화 리뷰의 감정을 분석합니다.',
    icon='search'
)

output_area = widgets.Output()

progress_bar = widgets.FloatProgress(
    value=0.0,
    min=0.0,
    max=1.0,
    description='신뢰도:',
    bar_style='info',
    orientation='horizontal',
    layout=widgets.Layout(width='90%')
)

# 5. 버튼 클릭 이벤트 콜백 함수 정의
def on_button_click(b):
    with output_area:
        clear_output()
        user_text = text_input.value.strip()
        if not user_text:
            print("오류: 감정을 분석할 텍스트를 입력해 주세요!")
            progress_bar.value = 0.0
            return
            
        # 감정 예측 실행
        pred_label, confidence, pos_prob, neg_prob = predict_sentiment(
            user_text, model, word_to_index, max_len=30, device=device
        )
        
        # 분석 결과를 예쁘게 포맷팅하여 출력
        print(f"입력 문장: \"{user_text}\"\n")
        if pred_label == 1:
            print(f"결과: 긍정 😊 (신뢰도: {confidence*100:.2f}%)")
            progress_bar.value = confidence
            progress_bar.bar_style = 'success'  # 초록색 게이지
        else:
            print(f"결과: 부정 😢 (신뢰도: {confidence*100:.2f}%)")
            progress_bar.value = confidence
            progress_bar.bar_style = 'danger'   # 빨간색 게이지
            
        print(f"\n[확률 정보] 긍정 확률: {pos_prob*100:.1f}% | 부정 확률: {neg_prob*100:.1f}%")

analyze_button.on_click(on_button_click)

# 6. 최종 레이아웃 배치 및 출력
ui_box = widgets.VBox([
    text_input,
    analyze_button,
    widgets.Label(value="* 분석 결과:"),
    output_area,
    progress_bar
], layout=widgets.Layout(padding='15px', border='1px solid #ddd', border_radius='5px', margin='10px 0'))

display(ui_box)